# Instance-Aware Depth on Colab

Runs the project's code on a Colab GPU. **Runtime → Change runtime type → T4 GPU** first.

**Data** (first match wins):
1. `MyDrive/instance-aware-depth/iad_data.tar` (your own copy), or
2. `MyDrive/iad_data.tar`: a **shortcut to the team's shared, view-only bundle**. In Drive, open
   "Shared with me", right-click `iad_data.tar` → Organize → Add shortcut → My Drive.
3. Neither: the notebook downloads and prepares everything (~10 min) and saves your own copy.

Checkpoints and results are always saved to **your own** `MyDrive/instance-aware-depth/`, never to the shared file.

**Private repo:** add a GitHub token as a Colab secret named `GH_TOKEN` (🔑 icon on the left).
Not needed once the repo is public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 1. Settings and Google Drive

In [9]:
import os
REPO = 'csgomez25/instance-aware-depth'
BRANCH = 'main'
DRIVE_DIR = '/content/drive/MyDrive/instance-aware-depth'   # YOUR folder: checkpoints, results, own data copy

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['DATA_ROOT'] = '/content/data'                     # local disk: much faster than reading from Drive
os.environ['CKPT_DIR'] = f'{DRIVE_DIR}/checkpoints'
os.environ['RUNS_DIR'] = f'{DRIVE_DIR}/runs'                # training runs survive disconnects

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Get the code

In [10]:
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir('/content/iad'):
    !git clone -q -b {BRANCH} {url} /content/iad
else:
    !git -C /content/iad pull -q
%cd /content/iad
!git log --oneline -1

/content/iad
c636c34 (HEAD -> main, origin/main, origin/HEAD) Eval: boundary F1, object-boundary F1 and flying-pixel metrics


In [11]:
!pip install -q -r requirements.txt
!apt-get -qq install -y rsync > /dev/null

## 3. Data: restore the bundle from Drive, or build it once

In [16]:
own_bundle = f'{DRIVE_DIR}/iad_data.tar'
shared_bundle = '/content/drive/MyDrive/iad_data.tar'          # shortcut to the team's view-only copy
bundle = next((b for b in (own_bundle, shared_bundle) if os.path.exists(b)), None)
os.makedirs(os.environ['DATA_ROOT'], exist_ok=True)
if bundle:
    print('restoring data from', bundle)
    data_root = os.environ['DATA_ROOT']
    !tar -xf "{bundle}" -C "{data_root}"                       # read-only: the bundle is never modified
!bash scripts/setup.sh      # fresh: downloads + prepares everything; restored: only code + checkpoints
if not bundle:
    !bash scripts/pack_data.sh "{own_bundle}"                  # saved to your own Drive folder only
!ls $DATA_ROOT $DATA_ROOT/nyu

restoring data from /content/drive/MyDrive/instance-aware-depth/iad_data.tar
== 1/6 Depth Anything V2 code (pinned to a561b84)
== 2/6 Checkpoints
== 3/6 DA-2K (~680 MB)
== 4/6 iBims-1 core (~55 MB, rsync from TUM; public password m1455541)
== 5/6 NYU Depth V2 labeled (~2.9 GB) -> image files
== 6/6 YOLO11s-seg masks
yolo11s-seg: 0it [00:00, ?it/s]
Masks for 1449 images in /content/data/nyu/yolo11s-seg
Setup done. Data in /content/data, checkpoints in /content/drive/MyDrive/instance-aware-depth/checkpoints.
/content/data:
DA-2K  ibims  nyu

/content/data/nyu:
depth_filled  instances  meta.csv  yolo11s-seg
depth_raw     labels	 rgb	   yolo11s-seg.jsonl


## 4. Smoke test (~1 min)

In [ ]:
%cd /content/iad/tools
!python infer.py $DATA_ROOT/nyu/rgb/000500.png --outdir /content/iad/results/infer
!python eval_nyu.py --split val --limit 10 --tag colab_smoke

## 5. Baseline evaluations (optional, ~5 min)
Use `--split val` while tuning; `--split test` only for final numbers.

In [ ]:
!python eval_nyu.py --split val  --gt filled
!python eval_nyu.py --split test --gt filled
!python eval_nyu.py --split test --gt raw
!python eval_da2k.py

## 6. Training
Runs are saved to `MyDrive/instance-aware-depth/runs/<name>/` (`log.csv`, `best.pth`).
If Colab disconnects, reconnect, run sections 1–3 again, then **re-run the same training cell**: it resumes
from the last finished epoch. A finished run just prints its result. Use a new `--name` for new settings.

Each run uses the val split for model selection; the test split is not touched.

In [ ]:
%cd /content/iad/tools
# Quick check that training works on this GPU (~1 min); prints speed and memory per epoch.
!python train_nyu.py --name colab_smoke --epochs 1 --max-steps 20 --bs 8

### H1: baseline fine-tune, filled vs raw depth target (decides the target for every later run)

In [ ]:
!python train_nyu.py --name h1_silog_filled --train-gt filled --bs 4 --epochs 20

In [ ]:
!python train_nyu.py --name h1_silog_raw --train-gt raw --bs 4 --epochs 20

### H2: instance-aware loss ablations (after H1; set TARGET to the H1 winner)
Each line is one run; re-running the cell skips finished runs and resumes an interrupted one.

In [ ]:
TARGET = 'filled'   # <- H1 winner: 'filled' or 'raw'
M = 'yolo11s-seg'
!python train_nyu.py --name h2_edge_yolo_l0.5    --train-gt {TARGET} --masks {M}  --lambda-edge 0.5
!python train_nyu.py --name h2_obj_yolo_l0.5     --train-gt {TARGET} --masks {M}  --lambda-obj 0.5
!python train_nyu.py --name h2_both_yolo_l0.5    --train-gt {TARGET} --masks {M}  --lambda-edge 0.5 --lambda-obj 0.5
!python train_nyu.py --name h2_both_gt_l0.5      --train-gt {TARGET} --masks gt   --lambda-edge 0.5 --lambda-obj 0.5

In [18]:
%cd /content/iad/tools
import os; print(os.environ.get('RUNS_DIR'))
!python train_nyu.py --name h2_silog_yolo_l0 --train-gt filled --masks yolo11s-seg

/content/iad/tools
/content/drive/MyDrive/instance-aware-depth/runs
xFormers not available
xFormers not available
h2_silog_yolo_l0: 703 train / 92 val images, 175 steps/epoch, bs 4, lr 5e-06, target filled, amp True, device cuda, masks yolo11s-seg, lambda_edge 0.0, lambda_obj 0.0
epoch 1/20  loss 0.1538  val AbsRel 0.1042 (raw 0.0966)  d1 0.9122  best 0.1042  [42s train, 11s val, 1.95 GB]
epoch 2/20  loss 0.1249  val AbsRel 0.1037 (raw 0.0942)  d1 0.9086  best 0.1037  [41s train, 11s val, 1.95 GB]
epoch 3/20  loss 0.1107  val AbsRel 0.1267 (raw 0.1193)  d1 0.8842  best 0.1037  [48s train, 11s val, 1.95 GB]
epoch 4/20  loss 0.1039  val AbsRel 0.0931 (raw 0.0857)  d1 0.9298  best 0.0931  [45s train, 11s val, 1.95 GB]
epoch 5/20  loss 0.0981  val AbsRel 0.0987 (raw 0.0917)  d1 0.9294  best 0.0931  [47s train, 11s val, 1.95 GB]
epoch 6/20  loss 0.0931  val AbsRel 0.0979 (raw 0.0899)  d1 0.9187  best 0.0931  [45s train, 11s val, 1.95 GB]
epoch 7/20  loss 0.0873  val AbsRel 0.0942 (raw 0.085

### H3: learning rate x batch size sweep on the SiLog baseline (one machine for the whole sweep)

In [ ]:
TARGET = 'filled'   # <- H1 winner
for lr in ['2e-6', '5e-6', '1e-5']:
    for bs in [2, 4, 8]:
        !python train_nyu.py --name h3_silog_lr{lr}_bs{bs} --train-gt {TARGET} --lr {lr} --bs {bs}

In [13]:
# Compare runs: best epoch (optimistic: lowest of many noisy scores) and mean of the last 5 epochs (steadier)
import glob, json, pandas as pd
rows = []
for f in sorted(glob.glob(f"{os.environ['RUNS_DIR']}/*/log.csv")):
    log = pd.read_csv(f); name = f.split('/')[-2]
    b = log.loc[log.val_filled_abs_rel.idxmin()]
    last5 = log.tail(5)
    rows.append({'run': name, 'epochs_done': len(log), 'best_epoch': int(b.epoch),
                 'best_val_abs_rel': b.val_filled_abs_rel, 'best_val_abs_rel_raw': b.val_raw_abs_rel, 'best_val_d1': b.val_filled_d1,
                 'last5_val_abs_rel': round(last5.val_filled_abs_rel.mean(), 4),   # steadier than the single best epoch
                 'last5_val_d1': round(last5.val_filled_d1.mean(), 4)})
pd.DataFrame(rows).sort_values('last5_val_abs_rel')

,run,epochs_done,best_epoch,best_val_abs_rel,best_val_abs_rel_raw,best_val_d1,last5_val_abs_rel,last5_val_d1
1,h1_silog_filled,20,15,0.0889,0.0802,0.9335,0.0910,0.9309
4,h2_both_yolo_l0.5,20,9,0.0895,0.0815,0.9317,0.0911,0.9329
6,h2_obj_yolo_l0.5,20,9,0.0882,0.0803,0.9328,0.0911,0.9325
3,h2_both_gt_l0.5,20,15,0.0893,0.0811,0.9309,0.0917,0.9308
5,h2_edge_yolo_l0.5,20,9,0.0903,0.0823,0.9302,0.0919,0.9303
2,h1_silog_raw,20,10,0.0914,0.0797,0.9306,0.0944,0.9256
0,colab_smoke,1,1,0.1419,0.1351,0.8325,0.1419,0.8325


In [20]:
%cd /content/iad/tools
import os
RD = os.environ['RUNS_DIR']
for run in ['h1_silog_filled', 'h2_silog_yolo_l0', 'h2_edge_yolo_l0.5', 'h2_obj_yolo_l0.5', 'h2_both_yolo_l0.5', 'h2_both_gt_l0.5']:
    ckpt = f"{RD}/{run}/best.pth"
    !python eval_nyu.py --split val --gt filled --ckpt "{ckpt}" --tag {run}

/content/iad/tools
xFormers not available
xFormers not available
100% 92/92 [00:21<00:00,  4.27it/s]
            tag split     gt  n       ms  abs_rel  sq_rel   rmse  rmse_log  log10     d1     d2     d3    bf1  bf1_obj  flying
h1_silog_filled   val filled 92 109.1848   0.0889  0.0603 0.3621    0.1156 0.0366 0.9335 0.9875 0.9952 0.1196   0.1506  0.5915
xFormers not available
xFormers not available
100% 92/92 [00:20<00:00,  4.42it/s]
             tag split     gt  n       ms  abs_rel  sq_rel   rmse  rmse_log  log10     d1     d2     d3    bf1  bf1_obj  flying
h2_silog_yolo_l0   val filled 92 108.5995   0.0889  0.0606 0.3628    0.1157 0.0366 0.9334 0.9875 0.9952 0.1186   0.1491  0.5932
xFormers not available
xFormers not available
100% 92/92 [00:20<00:00,  4.49it/s]
              tag split     gt  n       ms  abs_rel  sq_rel   rmse  rmse_log  log10     d1     d2     d3    bf1  bf1_obj  flying
h2_edge_yolo_l0.5   val filled 92 105.8671   0.0903  0.0587 0.3701    0.1181 0.0381 0.9302 0.988

## 7. Final test numbers (only once experiments are frozen)

In [ ]:
RUN = 'h1_silog_filled'
ckpt = f"{os.environ['RUNS_DIR']}/{RUN}/best.pth"
!python eval_nyu.py --split test --gt filled --ckpt "{ckpt}" --tag {RUN}
!python eval_nyu.py --split test --gt raw    --ckpt "{ckpt}" --tag {RUN}

## 8. Save results to Drive

In [22]:
import shutil, time
dst = f'{DRIVE_DIR}/results/{time.strftime("%Y%m%d-%H%M%S")}'
shutil.copytree('/content/iad/results', dst, ignore=shutil.ignore_patterns('*.ply', '*.npy'))
print('saved to', dst)

saved to /content/drive/MyDrive/instance-aware-depth/results/20261124-081213
